In [3]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv pyarrow fastparquet

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/695.2 kB ? eta -:--:--
   ---------------------------------------- 695.2/695.2 kB 14.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 16.2 MB/s  0:00:00

   ---------------------------------------- 0/6 [psycopg2-binary]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\tech\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [5]:
load_dotenv(dotenv_path='../.env')

user = os.getenv('DB_USER', 'postgres')
password = os.getenv('DB_PASSWORD', 'mysecretpassword')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'olist')

In [6]:
engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{db_name}')

In [8]:
orders = pd.read_sql("SELECT * FROM orders;", engine)
customers = pd.read_sql("SELECT * FROM customers;", engine)
order_items = pd.read_sql("SELECT * FROM order_items;", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments;", engine)
order_reviews = pd.read_sql("SELECT * FROM order_reviews;", engine)
products = pd.read_sql("SELECT * FROM products;", engine)
sellers = pd.read_sql("SELECT * FROM sellers;", engine)
print("DONE")

DONE


In [9]:
items_agg = order_items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    total_items=('order_item_id', 'count'),
    seller_id=('seller_id', 'first'),
    product_id=('product_id', 'first')
).reset_index()


In [10]:
payments_agg = order_payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    primary_payment_type=('payment_type', 'first')
).reset_index()

In [11]:
reviews_clean = order_reviews.groupby('order_id').first().reset_index()

print("DONE")

DONE


In [12]:
ml_df = orders.merge(customers, on='customer_id', how='left')
ml_df = ml_df.merge(items_agg, on='order_id', how='left')
ml_df = ml_df.merge(payments_agg, on='order_id', how='left')
ml_df = ml_df.merge(reviews_clean[['order_id', 'review_score']], on='order_id', how='left')
ml_df = ml_df.merge(products, on='product_id', how='left')
ml_df = ml_df.merge(sellers, on='seller_id', how='left')

In [13]:
# 
# 
#os.makedirs('../artifacts', exist_ok=True)
#output_path = '../artifacts/ml_table.parquet'
#ml_df.to_parquet(output_path, index=False)

In [ ]:

os.makedirs('../data/raw', exist_ok=True)
output_path = '../data/raw/ml_table.parquet'
ml_df.to_parquet(output_path, index=False)

In [16]:

ml_df[['order_id', 'customer_id', 'total_price', 'total_payment', 'review_score']].head(2)

,order_id,customer_id,total_price,total_payment,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,29.99,38.71,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,118.70,141.46,4.0
